## Notebook Overview

The following notebook provides a practical walkthrough of uploading imagery and labels to **CVDMS** using the programmatic API.

Before running the examples, ensure that:

1. The **CDK application has been deployed** to your AWS account.
2. You have ran the dataset bootstrapping code in _dataset_bootstrap_/ and produced manifests and images now present in **Amazon S3**.
Or, you have provided your own images and manifests in the required format.

---

## Supported Task Types

CVDMS currently supports five computer vision task types:

1. **Single Label Classification** (`label_type = "single-label"`)
2. **Multi-Label Classification** (`label_type = "multi-label"`)
3. **Object Detection** (`label_type = "object-detection"`)
4. **Semantic Segmentation** (`label_type = "semantic-segmentation"`)
5. **Instance Segmentation** (`label_type = "instance-segmentation"`)

In the examples below, one of these task types is selected and the sample manifests in the `samples/` directory are used as input. Either the **CSV** or **JSONL** version of the manifest may be referenced.
The sample CSV and JSONL files in the `samples/` folder may contain
image label pairs with no labels or duplicate entries, simply for testing
the capability of the code to detect such cases.


---

## Purpose of This Notebook

These examples are **not training workflows**. Their purpose is to demonstrate how imagery and labels are ingested into the CVDMS infrastructure.

The manifests follow the expected input formats described in **`README_upload.md`**, which documents the standardized schema used by the system (compatible with AWS Ground Truth–style annotation outputs).

Running this notebook allows you to:

- validate that the infrastructure was deployed correctly
- test ingestion of different label types
- verify the upload workflow and data normalization process

---

## Monitoring Jobs and Logs

A logging client is included that can return workflow logs as a **pandas DataFrame**, and an example usage is shown later in this notebook.

However, for inspecting logs and system state, it is generally easier to use the AWS Console:

- **Amazon Athena** for querying logs and Iceberg tables
- **AWS Step Functions** for monitoring workflow execution and debugging failures

These tools provide a clearer view of the upload pipeline and system state.

### Authenticate with AWS

Before interacting with the deployed infrastructure, you must authenticate with AWS and obtain temporary credentials.

The code cell below performs an AWS SSO login using the profile configured on your local machine.

Ensure that:

- You have created a user in **AWS IAM Identity Center (SSO)**.
- The user is assigned to the AWS account where the **CDK application was deployed**.
- The assigned role has permissions to access the CVDMS infrastructure.
- The profile you specify exists in your local AWS configuration (`~/.aws/config`) and points to the correct **account, role, and region**.

Running the command will open a browser window for authentication if your credentials are not already cached.

- `app_name` – the name used when deploying the CDK application
- `profile_name` – the AWS CLI profile used for authentication

In [ ]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid from your previous login.
!aws sso login --profile {profile_name}

### Initialize the CVDMS Client

The code below imports the programmatic API entry point and creates a client instance used to interact with the CVDMS infrastructure.

The client uses the AWS profile you authenticated with in the previous step and loads the necessary configuration (such as bucket names and table names) from AWS Systems Manager Parameter Store.

In [ ]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

### Start an Upload Job

The code below initiates the **CVDMS upload workflow** using the manifest provided in the `samples/` directory.

The `start_upload_job()` method uploads the manifest to S3 and triggers the ingestion pipeline described in `README_upload.md`.

Parameters used in this example:

- **`manifest_path`**
  Path to the local manifest file containing the image references and labels to ingest.

- **`label_type`**
  Specifies the task type for the upload. Must be one of the supported values:
  `single-label`, `multi-label`, `object-detection`, `semantic-segmentation`, or `instance-segmentation`.

- **`data_source`**
  A descriptive identifier stored in the `canonical_imagery` Iceberg table indicating where the imagery originated (for example `COCO`, `ImageNet`, etc.).

- **`path_prefix`**
  The storage prefix used when copying imagery into the canonical file bucket.
  Images are stored under:
   `canonical/imagery/<path_prefix>/`

- **`job_summary`**
A short description of the upload job. This text is stored in the DynamoDB job table and can help identify the purpose of the upload later.

Running this call uploads the manifest, creates a job entry in DynamoDB, and triggers the **Step Functions upload workflow**.

In [ ]:
manifest_path = r"C:\cvdms_files\sample_manifests\semantic_segmentation\coco2017_train\manifest.jsonl"
label_type = "semantic-segmentation"
data_source = "eurosat" # eurosat, coco2017, bigearthnetv2, ...
source_split = "train" # "train", "val", "test", None
path_prefix = "semantic-segmentation-images/classes"
job_summary = "Upload some samples: testing 1."

upload_info = app.start_upload_job(manifest_path,
                                   label_type,
                                   data_source,
                                   source_split,
                                   path_prefix,
                                   job_summary)

### Verify Job Submission

Run this cell to confirm that the upload job was submitted successfully.

If the submission succeeds, the returned response will contain a `job_id`.
This identifier can be used to monitor the workflow execution, inspect logs, and query job status.

If the submission fails, the returned error message will be displayed and additional details can be found in the local API log file located in `cvdms_platform/api_logs`.

In [ ]:
job_id = upload_info.get("job_id")
upload_error = upload_info.get("error")

if job_id:
    print(f"Upload job submitted successfully. Job ID: {job_id}")
else:
    print(f"Upload job submission failed. Error: {upload_error}")
    print("See local API logs in cvdms_platform/api_logs for more details.")

### Retrieve Workflow Logs

Use the log client to retrieve logs associated with the upload job.
Logs are stored centrally and queried using **Amazon Athena**, and the results are returned as a **pandas DataFrame**.

Note that log records may take a short time to appear because they are delivered through the logging pipeline before becoming queryable in Athena.

The cell below retrieves logs for the current `job_id` and prints the first few entries.

In [ ]:
log_info = app.get_logs_by_job_id(job_id)

if not log_info.get("error"):
    log_df = log_info["logs_df"]
    print("First 10 log entries:")
    print(log_df.head(10))
else:
    log_retrieval_error = log_info.get("error")
    print(f"Error retrieving logs: {log_retrieval_error}")

The upload job is now running asynchronously through the Step Functions workflow.
You can monitor execution progress in the **AWS Step Functions console** or query
logs and system tables using **Amazon Athena**.